In [5]:
# Install llama-cpp-python (with GPU support if Colab GPU is enabled)
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 GB 579.5 kB/s eta 0:00:00
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
Using cached diskcache-5.6.3-py3-none-any.whl (45 kB)


In [7]:
from llama_cpp import Llama

llm = Llama(
    model_path="/content/model2_retention_0.5bv2.gguf",
    n_ctx=4096,
    n_gpu_layers=-1,  # offload all layers to GPU; set 0 for CPU-only
    verbose=False
)

In [11]:
import json

with open("/content/test_modified.jsonl") as f:
    test_data = [json.loads(line) for line in f]  # Reads each line as a separate JSON object

print(f"Loaded {len(test_data)} test examples")
print(test_data[0])  # sanity check the format

Loaded 120 test examples
{'messages': [{'role': 'system', 'content': "You are a banking retention AI. Analyze the customer data and output strict JSON containing 'why', a churn probability field, and 'next_actions' as an object mapping each recommendation to an explanation."}, {'role': 'user', 'content': {'analysis_type': 'cluster', 'cluster_metadata': {'cluster_id': 'C_118', 'cluster_size': 119}, 'aggregated_data': {'dominant_profile': {'avg_age': 45, 'customer_segment': 'wealth'}, 'friction_signals': {'avg_complaints_30d': 0.2, 'avg_failed_transactions_30d': 1.1, 'common_complaint_themes': []}}, 'average_churn_probability': 0.652}}, {'role': 'assistant', 'content': {'why': "High-value cluster exhibiting synchronized capital flight with minimal complaints, strongly suggesting they are being targeted by a competitor's aggressive rate campaign.", 'next_actions': {'rate_offer': "Offer a competitive rate or renewal incentive for the customer's maturing FD, prioritizing a tailored retentio

In [21]:
import re
import time

# Updated ALLOWED_PREFIXES based on common action names in the dataset
ALLOWED_PREFIXES = [
    "rate_offer", "rm_call", "complaint_escalation", "product_upsell",
    "customer_survey", "fraud_alert", "tech_support_ticket", "proactive_call",
    "transaction_review", "security_review", "reward_points_adjustment",
    "loan_restructuring", "card_limit_increase", "cross_sell"
]

def extract_numbers(text):
    return set(re.findall(r'-?\d+\.?\d*', text))

results = []

for example in test_data:
    # Ensure all 'content' fields in messages are strings
    messages_for_llm = []
    for message in example["messages"]:
        if isinstance(message["content"], dict):
            messages_for_llm.append({
                "role": message["role"],
                "content": json.dumps(message["content"])
            })
        else:
            messages_for_llm.append(message)

    prompt = messages_for_llm
    input_numbers = extract_numbers(json.dumps(example))  # numbers present in the input record

    start = time.time()
    output = llm.create_chat_completion(messages=prompt, max_tokens=512, temperature=0.0)
    latency = time.time() - start

    raw_text = output["choices"][0]["message"]["content"].strip()

    # 1. JSON validity
    is_valid_json = False
    has_required_fields = False
    parsed = None
    try:
        parsed = json.loads(raw_text)
        is_valid_json = True
        has_required_fields = "why" in parsed and "next_actions" in parsed
    except json.JSONDecodeError:
        pass

    # 2. Prefix validity
    prefix_valid = None
    if has_required_fields:
        next_actions_data = parsed.get("next_actions")
        action_names = []
        if isinstance(next_actions_data, dict):
            action_names = list(next_actions_data.keys())
        elif isinstance(next_actions_data, list):
            action_names = [str(a) for a in next_actions_data]

        if action_names:
            valid_count = sum(
                1 for name in action_names
                if any(name.startswith(p) for p in ALLOWED_PREFIXES)
            )
            prefix_valid = valid_count / len(action_names)

    # 3. Grounding rate
    grounding = None
    if has_required_fields:
        why_numbers = extract_numbers(str(parsed.get("why", "")))
        if why_numbers:
            overlap = why_numbers & input_numbers
            grounding = len(overlap) / len(why_numbers)
        else:
            grounding = 1.0  # no numeric claims = trivially grounded (adjust if you disagree)

    results.append({
        "raw_output": raw_text,
        "json_valid": is_valid_json and has_required_fields,
        "prefix_validity": prefix_valid,
        "grounding_rate": grounding,
        "latency": latency
    })

In [22]:
n = len(results)

json_validity_pct = sum(r["json_valid"] for r in results) / n

prefix_scores = [r["prefix_validity"] for r in results if r["prefix_validity"] is not None]
prefix_validity_pct = sum(prefix_scores) / len(prefix_scores) if prefix_scores else 0

grounding_scores = [r["grounding_rate"] for r in results if r["grounding_rate"] is not None]
grounding_rate_pct = sum(grounding_scores) / len(grounding_scores) if grounding_scores else 0

mean_latency = sum(r["latency"] for r in results) / n

composite_score = (
    0.5 * json_validity_pct +
    0.3 * prefix_validity_pct +
    0.2 * grounding_rate_pct
)

print(f"JSON validity:    {json_validity_pct:.2%}")
print(f"Prefix validity:  {prefix_validity_pct:.2%}")
print(f"Grounding rate:   {grounding_rate_pct:.2%}")
print(f"Mean latency:     {mean_latency:.2f}s")
print(f"Composite score:  {composite_score:.4f}")

JSON validity:    97.50%
Prefix validity:  61.97%
Grounding rate:   98.29%
Mean latency:     0.43s
Composite score:  0.8700


In [23]:
with open("/content/eval_results.json", "w") as f:
    json.dump({
        "summary": {
            "json_validity": json_validity_pct,
            "prefix_validity": prefix_validity_pct,
            "grounding_rate": grounding_rate_pct,
            "mean_latency_sec": mean_latency,
            "composite_score": composite_score
        },
        "per_example": results
    }, f, indent=2)